In [1]:
from datasets import load_dataset

In [2]:
dataset = load_dataset("timm/oxford-iiit-pet", cache_dir="../data/hf_cache")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3680
    })
    test: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3669
    })
})

In [4]:
test_data = dataset["test"]

train_val_data = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_data = train_val_data["train"]
val_data = train_val_data["test"]

In [5]:
val_data.shape

(368, 4)

In [6]:
dataset["train"].shape

(3680, 4)

In [7]:
from transformers import ViTImageProcessor

model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(model_name)

In [8]:
target_size = processor.size["height"]
target_size

224

In [9]:
from torchvision.transforms import Compose, RandomResizedCrop, RandomHorizontalFlip

augment_transforms = Compose([
    RandomResizedCrop(target_size),
    RandomHorizontalFlip(),
])

augment_transforms

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
)

In [38]:
def transform_train(batch):
    augmented_images = [augment_transforms(img) for img in batch["image"]]

    inputs = processor(
        augmented_images,
        return_tensors="pt",
        do_resize=False, 
    )

    inputs["labels"] = batch["label"]

    return inputs

In [11]:
def transform_val(batch):
    images = [img for img in batch["image"]]
    inputs = processor(images, return_tensors="pt")
    
    inputs["labels"] = batch["label"]
    
    return inputs

In [39]:
train_data.set_transform(transform_train)
val_data.set_transform(transform_val)
test_data.set_transform(transform_val)

In [31]:
val_data[0]

{'pixel_values': tensor([[[-0.0902, -0.1216, -0.2078,  ..., -0.2471, -0.2392, -0.2314],
          [-0.0431, -0.0196, -0.0745,  ..., -0.2392, -0.2235, -0.2078],
          [-0.1608, -0.0431, -0.0039,  ..., -0.2314, -0.2235, -0.2235],
          ...,
          [-0.9059, -0.8902, -0.8902,  ...,  0.4980,  0.4980,  0.4902],
          [-0.9137, -0.9059, -0.8902,  ...,  0.4902,  0.4824,  0.4902],
          [-0.9216, -0.9059, -0.9059,  ...,  0.4745,  0.4745,  0.4824]],
 
         [[ 0.1216,  0.0824, -0.0353,  ..., -0.2863, -0.2784, -0.2706],
          [ 0.1529,  0.1765,  0.1529,  ..., -0.2706, -0.2549, -0.2471],
          [ 0.0980,  0.1765,  0.2000,  ..., -0.2549, -0.2549, -0.2471],
          ...,
          [-0.9059, -0.8980, -0.8980,  ...,  0.5451,  0.5451,  0.5451],
          [-0.9137, -0.9059, -0.8980,  ...,  0.5373,  0.5373,  0.5451],
          [-0.9216, -0.9137, -0.8980,  ...,  0.5216,  0.5216,  0.5373]],
 
         [[ 0.7412,  0.6549,  0.5059,  ..., -0.4353, -0.4275, -0.4196],
          [ 

In [13]:
labels = dataset["train"].features["label"].names
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}

In [14]:
import torch
import numpy as np
from transformers import ViTForImageClassification, TrainingArguments, Trainer, DefaultDataCollator

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
import evaluate

accuracy = evaluate.load("accuracy")

In [16]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [20]:
training_args = TrainingArguments(
    output_dir="../models/vit-pets",
    remove_unused_columns=False,    
    eval_strategy="epoch",    
    save_strategy="epoch",          
    learning_rate=5e-5,             
    per_device_train_batch_size=8,  
    gradient_accumulation_steps=4,  
    per_device_eval_batch_size=8,
    num_train_epochs=3,             
    warmup_steps=0.1,               
    logging_steps=10,               
    load_best_model_at_end=True,    
    metric_for_best_model="accuracy",
    push_to_hub=False,              
    report_to="none",               
    fp16=True,                      
    gradient_checkpointing=True,    
    dataloader_num_workers=2,      
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=processor,
    data_collator=DefaultDataCollator(),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()